In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer

In [2]:
# Load the data

data = pd.read_csv("training_data.csv")

In [3]:
#top few records
data.head()

,text,label
0,Wall St. Bears Claw Back Into the Black (Reute...,2
1,Carlyle Looks Toward Commercial Aerospace (Reu...,2
2,Oil and Economy Cloud Stocks' Outlook (Reuters...,2
3,Iraq Halts Oil Exports from Main Southern Pipe...,2
4,"Oil prices soar to all-time record, posing new...",2


In [4]:
# data size
data.shape

(120000, 2)

In [5]:
data["label"].value_counts()

label
2    30000
3    30000
1    30000
0    30000
Name: count, dtype: int64

In [6]:
# convert text to lowercase
text_lower = data["text"].apply(lambda x:x.strip().lower())

In [7]:
data_lower = data.copy()

data_lower["text"] = text_lower

In [8]:
data_lower.head()

,text,label
0,wall st. bears claw back into the black (reute...,2
1,carlyle looks toward commercial aerospace (reu...,2
2,oil and economy cloud stocks' outlook (reuters...,2
3,iraq halts oil exports from main southern pipe...,2
4,"oil prices soar to all-time record, posing new...",2


In [9]:
data_lower["text"] = data_lower["text"].replace(r"[^\w\s]","",regex=True)

In [10]:
data_lower["text_length"] = data_lower["text"].apply(lambda sen:len(sen.split()))

In [11]:
tokenizer = Tokenizer(num_words=10000)
tokenizer.fit_on_texts(data_lower["text"])

In [12]:
word_indexes = tokenizer.index_word

In [13]:
reversed_word_indexes = {word:index for index, word in word_indexes.items()}

In [14]:
sequences = tokenizer.texts_to_sequences(data_lower["text"])

In [15]:
data_lower["text_length"].quantile([0.25,0.50,0.75,0.90,0.99])

0.25    31.0
0.50    37.0
0.75    42.0
0.90    48.0
0.99    69.0
Name: text_length, dtype: float64

In [16]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

max_sequence_len = 70

X = pad_sequences(sequences, maxlen=max_sequence_len, padding="post")
y = data_lower["label"]

In [17]:
vocab_size = len(word_indexes)+1

In [18]:
data_lower.shape

(120000, 3)

In [19]:
data_lower["label"].unique()

array([2, 3, 1, 0])

In [20]:
from tensorflow.keras.utils import to_categorical

y = to_categorical(y, num_classes=4)

In [21]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2, random_state=42)

In [22]:
X_train.shape, y_train.shape

((96000, 70), (96000, 4))

In [23]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout

model = Sequential()
model.add(Embedding(input_dim=10000, output_dim=128, input_length=max_sequence_len))
model.add(Bidirectional(LSTM(64)))
model.add(Dropout(0.5))
model.add(Dense(64, activation="relu"))
model.add(Dense(4, activation="softmax"))


c:\Users\1997k\anaconda3\envs\news_clf\lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [24]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [25]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(monitor="val_loss",patience=3, restore_best_weights=True)

In [26]:
# model compilation

model.compile(optimizer="adam", loss="categorical_crossentropy",metrics=["accuracy"])

In [27]:
model.fit(X_train, y_train, epochs=3, batch_size=32,validation_data=(X_test, y_test),callbacks=[early_stop],verbose=1)

Epoch 1/3
3000/3000 ━━━━━━━━━━━━━━━━━━━━ 77s 25ms/step - accuracy: 0.8784 - loss: 0.3532 - val_accuracy: 0.9020 - val_loss: 0.2988
Epoch 2/3
3000/3000 ━━━━━━━━━━━━━━━━━━━━ 76s 25ms/step - accuracy: 0.9255 - loss: 0.2244 - val_accuracy: 0.9132 - val_loss: 0.2656
Epoch 3/3
3000/3000 ━━━━━━━━━━━━━━━━━━━━ 75s 25ms/step - accuracy: 0.9380 - loss: 0.1796 - val_accuracy: 0.9119 - val_loss: 0.2724


In [28]:
# save the model

model.save("bi_lstm.h5")

In [30]:
import pickle

with open("tokenizer.pkl","wb") as file:
    pickle.dump(tokenizer,file)

In [32]:
label_map = {
    0 : "World",
    1 : "Sports",
    2 : "Business",
    3 : "Sci/Tech" 
}

with open("label_mapper.pkl","wb") as file:
    pickle.dump(label_map, file)